# 02 — Model selection

**Phenomenological Growth Models Lab**

Seven models, one dataset. This notebook is about choosing between them — and about
why the model that fits best is often not the model you should use.

Assumes you have worked through [01 — Fitting basics](01_fitting_basics.ipynb).

## Setup

These notebooks run against the environment pinned in this repository's `Project.toml`
and `Manifest.toml`. The cell below activates it and installs anything missing.

The first run downloads and precompiles the dependencies — a few minutes, once.
Every run after that starts in seconds.

Because the versions are pinned in `Manifest.toml`, you get the same package versions
these notebooks were written against, whatever else is installed on your machine. If you
skip this cell and rely on your global environment, newer versions may resolve
differently and the numbers below may not match.

In [ ]:
using Pkg

# Works whether the notebook was launched from the repository root or from notebooks/.
root = isfile("Project.toml") ? "." : ".."
Pkg.activate(root)
Pkg.instantiate()

println("Active project: ", Base.active_project())

In [ ]:
using GrowthFit
using Plots, Statistics, Printf, DelimitedFiles, Random

default(size = (760, 420), lw = 2, legend = :topright, framestyle = :box,
        guidefontsize = 10, titlefontsize = 11)

# Threads. Notebooks 03 and 05 bootstrap in parallel. A Jupyter kernel does
# NOT inherit `-t` from whatever launched Jupyter — the thread count is fixed
# by the kernel itself — so if this says 1, see the README for the one-off
# `IJulia.installkernel` step. Results are identical either way; only speed
# changes.
println("GrowthFit loaded. Threads available: ", Threads.nthreads())
Threads.nthreads() == 1 && println("  (single-threaded — notebooks 03 and 05 will be slower; see README)");


> **Tip.** The bootstrap in notebooks 03 and 05 runs in parallel. If you want it faster,
> restart Julia with `-t 8`. Results are identical either way — each replicate is seeded
> independently of the thread count.

In [ ]:
function load_series(name)
    path = isfile(joinpath("data", name)) ? joinpath("data", name) :
           joinpath("..", "data", name)
    isfile(path) || error("Could not find $name. Run this notebook from inside the " *
                          "cloned repository so the data/ folder is reachable.")
    raw = readdlm(path, '\t', Float64)
    return raw[:, 1], raw[:, 2]          # time, incidence
end

timevect_all, ydata_all = load_series("jalisco_measles.txt")

@printf("%d weeks, peak of %.0f cases, %.0f cases total\n",
        length(ydata_all), maximum(ydata_all), sum(ydata_all))

In [ ]:
calibration_period = 40
timevect = timevect_all[1:calibration_period]
ydata    = ydata_all[1:calibration_period]
I0 = ydata[1]

## Fit everything

Seven models, same data, same error structure. This takes a minute.

In [ ]:
Random.seed!(1)

flags = (:exp, :ggm, :lm, :glm, :rich, :grm, :gom)
allfits = Dict{Symbol,Any}()

for flag in flags
    print("fitting $flag ... ")
    allfits[flag] = fit_growth_model(flag, timevect, ydata; dist = :nb1, n_restarts = 10)
    println(allfits[flag].converged ? "ok" : "FAILED")
end

## Goodness of fit is not enough

A more flexible model can never fit *worse* than one nested inside it — it can always
set the extra parameter to the value that reproduces the simpler model. So ranking by
likelihood alone would always pick the most complex model, whether or not the extra
flexibility is doing real work.

The standard correction is **AIC**, which charges two units of likelihood per parameter:

$$\mathrm{AIC} = 2k + 2\,\mathrm{NLL}$$

With short series — and 40 weeks is short — AIC under-penalizes, so we use the
small-sample correction **AICc**:

$$\mathrm{AICc} = \mathrm{AIC} + \frac{2k(k+1)}{n-k-1}$$

Lower is better. Only differences matter; the absolute value is meaningless.

In [ ]:
function aicc(fit, n; dist = :nb1)
    k = MODEL_NPARAMS[fit.flag] + (dist == :nb1 ? 1 : 0)   # +1 for alpha under NB1
    aic = 2k + 2 * fit.objective
    return aic + 2k * (k + 1) / (n - k - 1), k
end

n = length(ydata)
rows = [(flag, aicc(allfits[flag], n)..., allfits[flag].objective) for flag in flags]
sort!(rows, by = r -> r[2])

best = rows[1][2]
println("model                              k     NLL        AICc     ΔAICc")
println("-"^68)
for (flag, a, k, nll) in rows
    @printf("%-34s %d  %10.2f %10.2f %8.2f\n", MODEL_NAMES[flag], k, nll, a, a - best)
end

GrowthFit (0.3 and later) has this calculation built in as `compute_aicc`. Writing it
out by hand once is the point of the cell above; in your own work, use the package
function. Check they agree:

In [ ]:
for (flag, a, k, nll) in rows
    @printf("%-34s  by hand %10.2f   compute_aicc %10.2f\n",
            MODEL_NAMES[flag], a, compute_aicc(allfits[flag], :nb1))
end

They match here, under NB1 and under Poisson. They would **not** match under
`dist = :normal`. There the fit's objective is a sum of squared errors, not a negative
log-likelihood, and the formula above would give the wrong answer; `compute_aicc` uses
the least-squares form, $n \log(\mathrm{SSE}/n)$, in that case.

### Reading ΔAICc

The rule of thumb most people use (Burnham & Anderson):

- **Δ < 2** — substantial support; this model is competitive with the best
- **4 ≤ Δ ≤ 7** — considerably less support
- **Δ > 10** — essentially no support

If several models sit within 2 of each other, the data cannot distinguish them. That is
a finding, not a failure — and it is an argument for reporting more than one, or for
combining them into an ensemble.

## Look at the fits

In [ ]:
plt = scatter(timevect, ydata, label = "observed", mc = :grey55, ms = 4, msw = 0,
              xlabel = "week", ylabel = "weekly cases",
              title = "All seven models", legend = :topleft)
for flag in flags
    f = allfits[flag]
    f.converged && plot!(plt, timevect, f.fitcurve, label = String(flag), lw = 1.5)
end
plt

The exponential and generalized-growth models have no $K$, so they cannot turn over —
they climb forever. On data that contains the peak, that is fatal. The models with a
carrying capacity all trace the hump reasonably.

Same comparison on a log scale, where the failure of the unbounded models is starker:

In [ ]:
plt = scatter(timevect, max.(ydata, 0.5), label = "observed", mc = :grey55, ms = 4, msw = 0,
              yscale = :log10, xlabel = "week", ylabel = "weekly cases (log)",
              title = "Log scale", legend = :bottomright)
for flag in flags
    f = allfits[flag]
    f.converged && plot!(plt, timevect, max.(f.fitcurve, 0.5), label = String(flag), lw = 1.5)
end
plt

## The nesting structure

These models form a hierarchy. Each arrow means *is a special case of*:

```
  Exponential  ──(free p)──▶  Generalized Growth
       │                              │
   (add K)                        (add K)
       ▼                              ▼
   Logistic    ──(free p)──▶  Generalized Logistic
       │                              │
   (free a)                       (free a)
       ▼                              ▼
   Richards    ──(free p)──▶  Generalized Richards
```

So the Generalized Richards Model contains all the others except Gompertz. You can check
this numerically — fit GRM, then look at whether its extra parameters actually moved away
from the values that would collapse it to a simpler model:

In [ ]:
g = allfits[:grm]
@printf("GRM fitted p = %.4f   (p = 1 would make it a Richards model)\n", g.p_exp)
@printf("GRM fitted a = %.4f   (a = 1 would make it a Generalized Logistic)\n", g.a)
println()
@printf("Richards NLL             : %.2f\n", allfits[:rich].objective)
@printf("Generalized Logistic NLL : %.2f\n", allfits[:glm].objective)
@printf("GRM NLL                  : %.2f\n", g.objective)

If the extra parameters barely move from their collapsing values and the likelihood
barely improves, the extra flexibility bought nothing — and AICc will say so.

## What AICc will not tell you

Three things worth holding onto:

**1. It is in-sample.** AICc estimates how well the model would predict *new data from
the same process*, using only the data you fitted. It is not a forecast evaluation.
Notebook 05 does that properly, and the ranking is not always the same.

**2. It says nothing about identifiability.** A model can win on AICc while its
individual parameters are hopelessly correlated — you get a good curve made of numbers
you cannot interpret. That is notebook 04.

**3. It compares only the models you fitted.** All seven could be wrong.

## Does the window change the answer?

Refit everything using only the first 25 weeks — before the peak — and see whether the
ranking holds.

In [ ]:
Random.seed!(1)
tv25, yd25 = timevect_all[1:25], ydata_all[1:25]
fits25 = Dict(f => fit_growth_model(f, tv25, yd25; dist = :nb1, n_restarts = 10) for f in flags)

rows25 = [(f, aicc(fits25[f], length(yd25))[1]) for f in flags]
sort!(rows25, by = r -> r[2])
best25 = rows25[1][2]

println("Ranking on 25 weeks (pre-peak)        ΔAICc")
println("-"^50)
for (f, a) in rows25
    @printf("%-36s %8.2f\n", MODEL_NAMES[f], a - best25)
end

Compare that ordering with the 40-week one. Before the peak there is little information
about $K$ — the curve has not turned over yet — so models that need a carrying capacity
have less to distinguish them, and the unbounded ones look relatively better than they
deserve to.

This is the central practical difficulty of real-time forecasting. The moment you most
need a final-size estimate is the moment the data supports it least.

---

## Exercises

1. Repeat the whole notebook with `dist = :poisson`. Does the AICc ranking change?
   (It should — and think about why: Poisson has one fewer parameter, so $k$ drops.)

2. Fit only the models with a carrying capacity (`:lm`, `:glm`, `:rich`, `:grm`) and
   compare their estimates of $K$. How much do they disagree?

3. Run the 25-week comparison at 30 and 35 weeks too. At what point does the ranking
   stabilize? (Notebook 06 answers this systematically, one week at a time.)

**Next:** [03 — Uncertainty](03_uncertainty.ipynb), where the point estimates get
confidence intervals and we score the forecasts properly.